# Task 2: Graph Modeling & Centrality Analysis
In this notebook, we perform community detection, centrality metrics calculation, and core-periphery structure identification for each subreddit from the collected interactions data.


In [ ]:
import pandas as pd
import networkx as nx
import numpy as np
import os
import community.community_louvain as community_louvain # pip install python-louvain
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
print("Loading data...")
edges_df = pd.read_csv('graphs/all_edges_consolidated.csv')

# Aggregate the weights over all time windows for each subreddit
global_edges = edges_df.groupby(['Source', 'Target', 'Subreddit'], as_index=False)['Weight'].sum()

# Get the list of subreddits to process
subreddits = global_edges['Subreddit'].unique()
print(f"Found {len(subreddits)} subreddits.")
global_edges.head()

In [ ]:
def analyze_subreddit_graph(edges_group):
    # Construct a Directed Graph
    G_directed = nx.DiGraph()
    for _, row in edges_group.iterrows():
        G_directed.add_edge(row['Source'], row['Target'], weight=row['Weight'])
        
    # Remove self-loops
    G_directed.remove_edges_from(nx.selfloop_edges(G_directed))
    
    # 1. Centrality Metrics
    # Betweenness Centrality (using an unweighted approximation or ignoring weights for speed)
    # For large graphs, k=50 or 100 might be needed for approximation
    k = min(100, len(G_directed.nodes())) if len(G_directed.nodes()) > 500 else None
    betweenness = nx.betweenness_centrality(G_directed, k=k, weight='weight' if k is None else None)
    
    # PageRank
    try:
        pagerank = nx.pagerank(G_directed, weight='weight')
    except:
        pagerank = {node: 0 for node in G_directed.nodes()}
        
    # Construct an Undirected Graph for Louvain and Core-Periphery
    G_undirected = G_directed.to_undirected()
    
    # 2. Community Detection (Louvain)
    try:
        louvain_partition = community_louvain.best_partition(G_undirected, weight='weight')
    except:
        louvain_partition = {node: -1 for node in G_undirected.nodes()}
        
    # 3. Core-Periphery (k-core)
    core_numbers = nx.core_number(G_undirected)
    
    # Assemble into a DataFrame
    metrics = []
    for node in G_directed.nodes():
        metrics.append({
            'User': node,
            'Betweenness_Centrality': betweenness.get(node, 0),
            'PageRank': pagerank.get(node, 0),
            'Community_ID': louvain_partition.get(node, -1),
            'Core_Number': core_numbers.get(node, 0),
            'Degree': G_directed.degree(node),
            'In_Degree': G_directed.in_degree(node),
            'Out_Degree': G_directed.out_degree(node)
        })
        
    metrics_df = pd.DataFrame(metrics)
    return metrics_df, G_directed

In [ ]:
all_metrics = []

for subreddit in subreddits:
    print(f"Processing Subreddit: {subreddit}...")
    edges_group = global_edges[global_edges['Subreddit'] == subreddit]
    
    # Skip extremely small networks
    if len(edges_group) < 5:
        print("  - Not enough edges, skipping.")
        continue
        
    metrics_df, G_directed = analyze_subreddit_graph(edges_group)
    metrics_df['Subreddit'] = subreddit
    all_metrics.append(metrics_df)
    
final_metrics_df = pd.concat(all_metrics, ignore_index=True)

# Save to CSV
if not os.path.exists('graphs'):
    os.makedirs('graphs')
final_metrics_df.to_csv('graphs/node_metrics.csv', index=False)
print("Finished processing all subreddits and saved to 'graphs/node_metrics.csv'.")
 final_metrics_df.head()

In [ ]:
# Let's verify by finding the top 5 influencers (PageRank) per subreddit
top_influencers = final_metrics_df.sort_values(
    ['Subreddit', 'PageRank'], ascending=[True, False]
).groupby('Subreddit').head(5)

print("Top 5 users by PageRank in some subreddits:")
top_influencers[['Subreddit', 'User', 'PageRank', 'Betweenness_Centrality', 'Community_ID', 'Core_Number']].head(15)